In [1]:
import pandas as pd
import happybase

In [3]:
df = pd.read_csv("/home/jovyan/data/movie_success_rate.csv")

In [4]:
conn = happybase.Connection(
    host="hbase",
    port=9090,
    timeout=20000
)
conn.open()

if b'movie_success_rate_raw' not in conn.tables():
    conn.create_table(
        'movie_success_rate_raw',
        {'cf': dict()}
    )

table = conn.table('movie_success_rate_raw')

In [5]:
CHUNK_SIZE = 100

for start in range(0, len(df), CHUNK_SIZE):
    end = start + CHUNK_SIZE
    chunk = df.iloc[start:end]

    batch = table.batch() 

    for idx, row in chunk.iterrows():
        if pd.isna(row["Title"]):
            continue

        rowkey = f"{row['Title']}_{row['Year']}_{idx}".encode()

        batch.put(rowkey, {
            b'cf:title': str(row['Title']).encode(),
            b'cf:year': str(row['Year']).encode(),
            b'cf:rating': str(row['Rating']).encode(),
            b'cf:votes': str(row['Votes']).encode(),
            b'cf:revenue': str(row['Revenue (Millions)']).encode(),
            b'cf:metascore': str(row['Metascore']).encode(),
            b'cf:success': str(row['Success']).encode(),
        })

    batch.send()
    print(f"Inserted rows {start} to {end}")

Inserted rows 0 to 100
Inserted rows 100 to 200
Inserted rows 200 to 300
Inserted rows 300 to 400
Inserted rows 400 to 500
Inserted rows 500 to 600
Inserted rows 600 to 700
Inserted rows 700 to 800
Inserted rows 800 to 900


In [6]:
for key, data in table.scan(limit=5):
    print(key.decode(), data)

(500) Days of Summer {b'cf:metascore': b'76.0', b'cf:rating': b'7.7', b'cf:revenue': b'32.39', b'cf:success': b'0', b'cf:title': b'(500) Days of Summer', b'cf:votes': b'398972', b'cf:year': b'2009'}
(500) Days of Summer_2009.0_443 {b'cf:metascore': b'76.0', b'cf:rating': b'7.7', b'cf:revenue': b'32.39', b'cf:success': b'0.0', b'cf:title': b'(500) Days of Summer', b'cf:votes': b'398972.0', b'cf:year': b'2009.0'}
10 Cloverfield Lane {b'cf:metascore': b'76.0', b'cf:rating': b'7.2', b'cf:revenue': b'71.9', b'cf:success': b'0', b'cf:title': b'10 Cloverfield Lane', b'cf:votes': b'192968', b'cf:year': b'2016'}
10 Cloverfield Lane_2016.0_103 {b'cf:metascore': b'76.0', b'cf:rating': b'7.2', b'cf:revenue': b'71.9', b'cf:success': b'0.0', b'cf:title': b'10 Cloverfield Lane', b'cf:votes': b'192968.0', b'cf:year': b'2016.0'}
12 Years a Slave {b'cf:metascore': b'96.0', b'cf:rating': b'8.1', b'cf:revenue': b'56.67', b'cf:success': b'0', b'cf:title': b'12 Years a Slave', b'cf:votes': b'486338', b'cf:y